In [33]:
import os
import bisect
from retry.api import retry_call
from python.git_operate import get_repo
from tqdm.auto import tqdm
import pandas as pd


URL_TEMPLATE = "https://github.com/{name_with_owner}/commit/{commit}"


top_dir = os.path.join(os.path.abspath(""))
res_dir = os.path.join(top_dir, "results")
data_dir = os.path.join(res_dir, "data")
owners_path = os.path.join(res_dir, "{language}")
names_path = os.path.join(res_dir, "{language}/{owner}")
pkl_path = os.path.join(res_dir, "{language}/{name_with_owner}.pkl")


def __RQ4(language: str) -> pd.DataFrame:
    rq3_dir = os.path.join(res_dir, f"rq3/{language}")

    all_df = pd.DataFrame()

    csv_paths = list(filter(lambda x: x.endswith(".csv"), os.listdir(rq3_dir)))
    csv_paths.sort()

    for csv_path in csv_paths:
        csv_df = pd.read_csv(os.path.join(rq3_dir, csv_path))

        name_with_owner = csv_path.replace(".csv", "")

        java_res_df = pd.read_pickle(
            os.path.join(
                top_dir,
                f"results/{language}",
                f"{name_with_owner.replace('_', '/', 1)}.pkl",
            )
        )

        java_res_df = java_res_df.map(lambda x: x if x else -1)

        java_res_df = java_res_df.astype(
            {
                "src start pos": "int",
                "src end pos": "int",
                "dst start pos": "int",
                "dst end pos": "int",
            }
        )

        java_res_df.to_pickle(
            os.path.join(
                top_dir,
                f"results/{language}",
                f"{name_with_owner.replace('_', '/', 1)}.pkl",
            )
        )

        merged_df = (
            csv_df.merge(
                java_res_df[
                    [
                        "commit",
                        "dst file",
                        "src file",
                        "src start pos",
                        "src end pos",
                        "dst start pos",
                        "dst end pos",
                    ]
                ],
                left_on=[
                    "insert commit",
                    "insert dst file",
                    "insert dst start pos",
                    "insert dst end pos",
                ],
                right_on=["commit", "dst file", "dst start pos", "dst end pos"],
                how="inner",
            )
            .drop(columns=["commit", "dst file", "dst start pos", "dst end pos"])
            .rename(
                columns={
                    "src file": "insert src file",
                    "src start pos": "insert src start pos",
                    "src end pos": "insert src end pos",
                }
            )
        )

        merged_df = (
            merged_df.merge(
                java_res_df[
                    [
                        "commit",
                        "dst file",
                        "src file",
                        "src start pos",
                        "src end pos",
                        "dst start pos",
                        "dst end pos",
                    ]
                ],
                left_on=[
                    "delete commit",
                    "delete src file",
                    "delete src start pos",
                    "delete src end pos",
                ],
                right_on=[
                    "commit",
                    "src file",
                    "src start pos",
                    "src end pos",
                ],
                how="inner",
            )
            .drop(
                columns=[
                    "commit",
                    "src file",
                    "src start pos",
                    "src end pos",
                ]
            )
            .rename(
                columns={
                    "dst file": "delete dst file",
                }
            )
        )

        merged_df["name_with_owner"] = name_with_owner

        all_df = pd.concat([all_df, merged_df])

    all_df = all_df.reindex(
        columns=[
            "name_with_owner",
            "insert commit",
            "insert src file",
            "insert src start pos",
            "insert src end pos",
            "insert dst file",
            "insert dst start pos",
            "insert dst end pos",
            "delete commit",
            "delete src file",
            "delete src start pos",
            "delete src end pos",
            "delete dst file",
        ],
    )
    all_df.reset_index(inplace=True, drop=True)

    return all_df


def calculate_line_offsets(source_code: str) -> list[int]:
    """
    Calculates the start offsets for each line in the source code.

    Args:
        source_code (str): The full source code as a string.

    Returns:
        list[int]: A list of line start offsets.
    """
    offsets = [0]
    current_offset = 0

    for line in source_code.splitlines(keepends=True):
        current_offset += len(line)
        offsets.append(current_offset)

    return offsets


def position_for(offset: int, source_code: str) -> tuple[int, int]:
    """
    Converts an offset into a (line, column) position based on the source code.

    Args:
        offset (int): The offset in the source code string.
        source_code (str): The full source code as a string.

    Returns:
        tuple: (line, column), both starting from 1.
    """
    lines = calculate_line_offsets(source_code)
    line = bisect.bisect_right(lines, offset) - 1
    column = offset - lines[line] + 1

    return line + 1, column


def removeNonAscii(s):
    return "".join(c for c in s if ord(c) < 128 and c != "\r")


extensions = {
    "java": "java",
    "javascript": "js",
    "ruby": "rb",
    "php": "php",
    "cpp": "cpp",
    "csharp": "cs",
}


def rq4(language: str):
    extension = extensions[language]

    rq4_df = __RQ4(language)
    rq4_df = rq4_df[268:]

    res_df = pd.DataFrame()

    for i, row in tqdm(rq4_df.iterrows(), total=len(rq4_df)):
        name_with_owner = row["name_with_owner"]
        insert_commit = row["insert commit"]
        insert_src_file = row["insert src file"]
        insert_dst_file = row["insert dst file"]
        insert_src_start_pos = row["insert src start pos"]
        insert_src_end_pos = row["insert src end pos"]
        insert_dst_start_pos = row["insert dst start pos"]
        insert_dst_end_pos = row["insert dst end pos"]
        delete_commit = row["delete commit"]
        delete_src_file = row["delete src file"]
        delete_src_start_pos = row["delete src start pos"]
        delete_src_end_pos = row["delete src end pos"]
        delete_dst_file = row["delete dst file"]

        assert insert_dst_file == delete_src_file

        repo = get_repo(top_dir, name_with_owner.replace("_", "/", 1), language)

        insert_src_file_content = removeNonAscii(
            retry_call(
                repo.git.show,
                fargs=[f"{insert_commit}^:{insert_src_file}"],
                tries=5,
                delay=1,
            )
        )
        insert_src_file_path = os.path.join(
            top_dir, f"rq4/{language}/{i}_insert_src.{extension}"
        )

        insert_dst_file_content = removeNonAscii(
            retry_call(
                repo.git.show,
                fargs=[f"{insert_commit}:{insert_dst_file}"],
                tries=5,
                delay=1,
            )
        )
        insert_dst_file_path = os.path.join(
            top_dir, f"rq4/{language}/{i}_insert_dst.{extension}"
        )

        if (
            name_with_owner == "signumsoftware_framework"
            and insert_commit == "058f6bf4968cfb578ba5b41f6b1e9bc1b07f5341"
        ):
            print(
                len(
                    retry_call(
                        repo.git.show,
                        fargs=[f"{insert_commit}:{insert_dst_file}"],
                        tries=5,
                        delay=1,
                    )
                ),
                len(insert_dst_file_content),
            )
            return insert_dst_file_content

        delete_src_file_content = removeNonAscii(
            retry_call(
                repo.git.show,
                fargs=[f"{delete_commit}^:{delete_src_file}"],
                tries=5,
                delay=1,
            )
        )
        delete_src_file_path = os.path.join(
            top_dir, f"rq4/{language}/{i}_delete_src.{extension}"
        )

        delete_dst_file_content = removeNonAscii(
            retry_call(
                repo.git.show,
                fargs=[f"{delete_commit}:{delete_dst_file}"],
                tries=5,
                delay=1,
            )
        )
        delete_dst_file_path = os.path.join(
            top_dir, f"rq4/{language}/{i}_delete_dst.{extension}"
        )

        row.loc["insert url"] = URL_TEMPLATE.format(
            name_with_owner=name_with_owner.replace("_", "/", 1), commit=insert_commit
        )
        row.loc["delete url"] = URL_TEMPLATE.format(
            name_with_owner=name_with_owner.replace("_", "/", 1), commit=delete_commit
        )
        row.loc["insert src file content"] = insert_src_file_content[
            insert_src_start_pos:insert_src_end_pos
        ]
        row.loc["insert dst file content"] = insert_dst_file_content[
            insert_dst_start_pos:insert_dst_end_pos
        ]
        row.loc["delete src file content"] = delete_src_file_content[
            delete_src_start_pos:delete_src_end_pos
        ]
        row.loc["insert src start rc"] = position_for(
            insert_src_start_pos, insert_src_file_content
        )
        row.loc["insert dst start rc"] = position_for(
            insert_dst_start_pos, insert_dst_file_content
        )
        row.loc["delete src start rc"] = position_for(
            delete_src_start_pos, delete_src_file_content
        )

        insert_commit = repo.commit(insert_commit)
        row.loc["replacement_msg"] = insert_commit.message

        delete_commit = repo.commit(delete_commit)
        row.loc["delete_msg"] = delete_commit.message

        res_df = pd.concat([res_df, row.to_frame().T])

    res_df["coding"] = None
    res_df = res_df.reindex(
        columns=[
            "name_with_owner",
            "insert url",
            "delete url",
            "replacement_msg",
            "delete_msg",
            "coding",
            "insert commit",
            "insert src file",
            "insert src start pos",
            "insert src start rc",
            "insert dst file",
            "insert dst start pos",
            "insert dst start rc",
            "delete commit",
            "delete src file",
            "delete src start pos",
            "delete dst file",
            "delete dst start pos",
            "delete dst start rc",
            "insert src file content",
            "insert dst file content",
            "delete src file content",
        ],
    )

    res_df = res_df.sample(frac=1, random_state=0)
    res_df = res_df.reset_index(drop=True)

    res_df.to_csv(os.path.join(res_dir, f"rq4/{language}_shuffle2.csv"))

    return res_df

In [34]:
df = rq4("csharp")

  0%|          | 0/57 [00:00<?, ?it/s]

11583 11299


In [36]:
print(len(df))

11299


In [16]:
pd.set_option("display.max_columns", None)

df

,name_with_owner,insert url,delete url,replacement_msg,delete_msg,coding,insert commit,insert src file,insert src start pos,insert src start rc,insert dst file,insert dst start pos,insert dst start rc,delete commit,delete src file,delete src start pos,delete dst file,delete dst start pos,delete dst start rc,insert src file content,insert dst file content,delete src file content
0,signumsoftware_framework,https://github.com/signumsoftware/framework/co...,https://github.com/signumsoftware/framework/co...,elefantes\n\n(TFS CS 5989)\n,Chequinazo entitygroups\n\n(TFS CS 6663)\n,None,fe7d3ff8826e4f2dfae0a3085882f37f3d990d5c,Signum.Engine.Extensions/Authorization/EntityG...,4187,"(113, 7)",Signum.Engine.Extensions/Authorization/EntityG...,4182,"(115, 3)",eb1b533826ad0b786fe883bdabdda8823e246a1b,Signum.Engine.Extensions/Authorization/EntityG...,5317,Signum.Engine.Extensions/Authorization/EntityG...,NaN,NaN,if (!saveDisabled && args.WasModified)\r...,if (!saveDisabled && ident.Modified)...,aveOptions.Destiny))\r\n {\r\n ...
1,Ginger-Automation_Ginger,https://github.com/Ginger-Automation/Ginger/co...,https://github.com/Ginger-Automation/Ginger/co...,Changed async void to task\n,making AccountReport Logger method async\n,None,58eaea8958fe9336d1c1be586932b7785c4735dd,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,3841,"(98, 13)",Ginger/GingerCoreNET/Run/RunListenerLib/Center...,4139,"(101, 13)",79a02f5bfbdc412bf3872a44306b5d3876b855b3,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,4747,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,NaN,NaN,BusinessFlowStartTask(businessFlow);,Task.Run(() => BusinessFlowStartTask(businessF...,() => BusinessFlowStartTask(businessFlow)
2,AutoMapper_AutoMapper,https://github.com/AutoMapper/AutoMapper/commi...,https://github.com/AutoMapper/AutoMapper/commi...,Array mapping simplified quite a bit\n,"Instead of using ToArray, which double maps. F...",None,d8ea3389e0104fad7787964e369a1973697a0445,src/AutoMapper/Mappers/ArrayMapper.cs,1927,"(46, 11)",src/AutoMapper/Mappers/ArrayMapper.cs,1043,"(29, 13)",fb68f979b31f4875d636d18d0e6f6dbd6c74fc7b,src/AutoMapper/Mappers/ArrayMapper.cs,709,src/AutoMapper/Mappers/ArrayMapper.cs,NaN,NaN,return array as TDestinatio,return source.Cast<TSourceElement>()\n ...,"item => newItemFunc(item, itemContext)"
3,Ginger-Automation_Ginger,https://github.com/Ginger-Automation/Ginger/co...,https://github.com/Ginger-Automation/Ginger/co...,Changed async void to task\n,making AccountReport Logger method async\n,None,58eaea8958fe9336d1c1be586932b7785c4735dd,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,3273,"(84, 13)",Ginger/GingerCoreNET/Run/RunListenerLib/Center...,3555,"(87, 13)",79a02f5bfbdc412bf3872a44306b5d3876b855b3,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,4163,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,NaN,NaN,RunnerRunEndTask(gingerRunner);,Task.Run(() => RunnerRunEndTask(gingerRunner));,() => RunnerRunEndTask(gingerRunner)
4,signumsoftware_framework,https://github.com/signumsoftware/framework/co...,https://github.com/signumsoftware/framework/co...,TryToken and QueryTokenDN builder with error m...,SubTokens(canAggregate)\n,None,058f6bf4968cfb578ba5b41f6b1e9bc1b07f5341,Signum.Web.Extensions/Chart/ChartClient.cs,9371,"(220, 68)",Signum.Web.Extensions/Chart/ChartClient.cs,8210,"(199, 92)",9880339ef979a380e89089dacaab00ec86c3c093,Signum.Web.Extensions/Chart/ChartClient.cs,8356,Signum.Web.Extensions/Chart/ChartClient.cs,NaN,NaN,o(\r\n,"chartToken, IChartBase chart, QueryDescription...",{\r\n bool canAggregate = (c...
5,rsdn_CodeJam,https://github.com/rsdn/CodeJam/commit/589249f...,https://github.com/rsdn/CodeJam/commit/82bf950...,Operator templates: comparison operators\n,Cleanup\n,None,589249fd5376b80b5ce93518ddf4f015e1537303,Main/src/Arithmetic/Operators.cs,508,"(21, 3)",Main/src/Arithmetic/Operators.cs,560,"(21, 3)",82bf9506fe7e1b8c4587b048c9762888cc8210d3,Main/src/Arithmetic/Operators.cs,646,Main/src/Arithmetic/Operators.cs,NaN,NaN,"public static readonly

In [ ]:
df = pd.read_json(
    "/Users/kz/labo/study/fact_replacing_task_re/results/rq4/csharp_shuffle2.json"
)

df

,name_with_owner,insert url,delete url,replacement_msg,delete_msg,coding,insert commit,insert src file,insert src start pos,insert dst file,insert dst start pos,delete commit,delete src file,delete src start pos,delete dst file,delete dst start pos,insert src file content,insert dst file content,delete src file content
0,signumsoftware_framework,https://github.com/signumsoftware/framework/co...,https://github.com/signumsoftware/framework/co...,elefantes\n\n(TFS CS 5989)\n,Chequinazo entitygroups\n\n(TFS CS 6663)\n,NaN,fe7d3ff8826e4f2dfae0a3085882f37f3d990d5c,Signum.Engine.Extensions/Authorization/EntityG...,"[113, 7]",Signum.Engine.Extensions/Authorization/EntityG...,"[115, 3]",eb1b533826ad0b786fe883bdabdda8823e246a1b,Signum.Engine.Extensions/Authorization/EntityG...,"[130, 51]",Signum.Engine.Extensions/Authorization/EntityG...,NaN,if (!saveDisabled && args.WasModified)\r...,if (!saveDisabled && ident.Modified)...,aveOptions.Destiny))\r\n {\r\n ...
1,Ginger-Automation_Ginger,https://github.com/Ginger-Automation/Ginger/co...,https://github.com/Ginger-Automation/Ginger/co...,Changed async void to task\n,making AccountReport Logger method async\n,NaN,58eaea8958fe9336d1c1be586932b7785c4735dd,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,"[98, 13]",Ginger/GingerCoreNET/Run/RunListenerLib/Center...,"[101, 13]",79a02f5bfbdc412bf3872a44306b5d3876b855b3,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,"[119, 22]",Ginger/GingerCoreNET/Run/RunListenerLib/Center...,NaN,BusinessFlowStartTask(businessFlow);,Task.Run(() => BusinessFlowStartTask(businessF...,() => BusinessFlowStartTask(businessFlow)
2,AutoMapper_AutoMapper,https://github.com/AutoMapper/AutoMapper/commi...,https://github.com/AutoMapper/AutoMapper/commi...,Array mapping simplified quite a bit\n,"Instead of using ToArray, which double maps. F...",NaN,d8ea3389e0104fad7787964e369a1973697a0445,src/AutoMapper/Mappers/ArrayMapper.cs,"[46, 11]",src/AutoMapper/Mappers/ArrayMapper.cs,"[29, 13]",fb68f979b31f4875d636d18d0e6f6dbd6c74fc7b,src/AutoMapper/Mappers/ArrayMapper.cs,"[21, 34]",src/AutoMapper/Mappers/ArrayMapper.cs,NaN,return array as TDestinatio,return source.Cast<TSourceElement>()\n ...,"item => newItemFunc(item, itemContext)"
3,Ginger-Automation_Ginger,https://github.com/Ginger-Automation/Ginger/co...,https://github.com/Ginger-Automation/Ginger/co...,Changed async void to task\n,making AccountReport Logger method async\n,NaN,58eaea8958fe9336d1c1be586932b7785c4735dd,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,"[84, 13]",Ginger/GingerCoreNET/Run/RunListenerLib/Center...,"[87, 13]",79a02f5bfbdc412bf3872a44306b5d3876b855b3,Ginger/GingerCoreNET/Run/RunListenerLib/Center...,"[105, 22]",Ginger/GingerCoreNET/Run/RunListenerLib/Center...,NaN,RunnerRunEndTask(gingerRunner);,Task.Run(() => RunnerRunEndTask(gingerRunner));,() => RunnerRunEndTask(gingerRunner)
4,signumsoftware_framework,https://github.com/signumsoftware/framework/co...,https://github.com/signumsoftware/framework/co...,TryToken and QueryTokenDN builder with error m...,SubTokens(canAggregate)\n,NaN,058f6bf4968cfb578ba5b41f6b1e9bc1b07f5341,Signum.Web.Extensions/Chart/ChartClient.cs,"[220, 68]",Signum.Web.Extensions/Chart/ChartClient.cs,"[199, 92]",9880339ef979a380e89089dacaab00ec86c3c093,Signum.Web.Extensions/Chart/ChartClient.cs,"[199, 2]",Signum.Web.Extensions/Chart/ChartClient.cs,NaN,o(\r\n,"chartToken, IChartBase chart, QueryDescription...",{\r\n bool canAggregate = (c...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,signumsoftware_framework,https://github.com/signumsoftware/framework/co...,https://github.com/signumsoftware/framework/co...,(TFS CS 4096)\n\n,.Net Framework 4.0 y Visual Studio 2010\n\n(TF...,NaN,2f429cedfc8a45053cc6398c790bdf9f230d96eb,Signum.Windows.Extensions/Authorization/AuthCl...,"[31, 9]",Signum.Windows.Extensions/Authorization/AuthCl...,"[31, 9]",ce1be332034042b46828ed0e08ab4203b0ea3f01,Signum.Windows.Extensions/Authorization/AuthCl...,"[41, 7]",Signum.W

In [5]:
languages = [
    "java",
    "javascript",
    "cpp",
    "csharp",
    "php",
    "ruby",
]

for language in languages:
    rq4(language)

  0%|          | 0/172 [00:00<?, ?it/s]

  0%|          | 0/183 [00:00<?, ?it/s]

  0%|          | 0/164 [00:00<?, ?it/s]

  0%|          | 0/325 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
def calculate_line_offsets(source_code: str) -> list[int]:
    """
    Calculates the start offsets for each line in the source code.

    Args:
        source_code (str): The full source code as a string.

    Returns:
        list[int]: A list of line start offsets.
    """
    offsets = [0]
    current_offset = 0

    for line in source_code.splitlines(keepends=True):
        current_offset += len(line)
        offsets.append(current_offset)

    return offsets


def position_for(offset: int, source_code: str) -> tuple[int, int]:
    """
    Converts an offset into a (line, column) position based on the source code.

    Args:
        offset (int): The offset in the source code string.
        source_code (str): The full source code as a string.

    Returns:
        tuple: (line, column), both starting from 1.
    """
    lines = calculate_line_offsets(source_code)
    line = bisect.bisect_right(lines, offset) - 1
    column = offset - lines[line] + 1

    return line + 1, column


# サンプルのソースコード文字列
source_code = """public class Sample {


    public static void main(String[] args) {
        System.out.println("Hello, World!");
    }
}"""


def get_line_column(source: str, offset: int) -> tuple[int, int]:
    lines = source[:offset].splitlines()
    line_number = len(lines)
    column_number = len(lines[-1]) + 1 if lines else 1
    return line_number, column_number


# 使用例
offset = source_code.find("System.out")
print(f"Offset: {offset}")
print("Position:", position_for(offset, source_code))

print(get_line_column(source_code, offset))  # (3, 7)


# サンプルコード
source_code = (
    """def add(a, b):\n\n    return a + b\n# 44tomgrgrgmorvpov\nprint(add(2, 3))"""
)
offset = source_code.index("add(2, 3)")  # "add(2, 3)" の開始位置

print(f"Offset: {offset}")
print("Position:", position_for(offset, source_code))

print(get_line_column(source_code, offset))  # (3, 7)

Offset: 77
Position: (5, 9)
(5, 9)
Offset: 59
Position: (5, 7)
(5, 7)


In [24]:
print(source_code)

def add(a, b):

    return a + b
# 44tomgrgrgmorvpov
print(add(2, 3))


In [ ]:
import os
from retry.api import retry_call
from python.git_operate import get_repo
from tqdm.auto import tqdm

import subprocess

URL_TEMPLATE = "https://github.com/{name_with_owner}/commit/{commit}"

top_dir = os.path.join(os.path.abspath(""))


def removeNonAscii(s):
    return "".join(c for c in s if ord(c) < 128)


language = "java"
os.makedirs(os.path.join(res_dir, f"rq4/{language}"), exist_ok=True)

rq4_df = __RQ4(language)

res_df = pd.DataFrame()

for i, row in tqdm(rq4_df.iterrows(), total=len(rq4_df)):
    name_with_owner = row["name_with_owner"]
    insert_commit = row["insert commit"]
    insert_src_file = row["insert src file"]
    insert_dst_file = row["insert dst file"]
    insert_src_start_pos = row["insert src start pos"]
    insert_src_end_pos = row["insert src end pos"]
    insert_dst_start_pos = row["insert dst start pos"]
    insert_dst_end_pos = row["insert dst end pos"]
    delete_commit = row["delete commit"]
    delete_src_file = row["delete src file"]
    delete_start_pos = row["delete src start pos"]
    delete_end_pos = row["delete src end pos"]
    delete_dst_file = row["delete dst file"]

    assert insert_dst_file == delete_src_file

    repo = get_repo(top_dir, name_with_owner.replace("_", "/", 1), language)

    insert_src_file_content = removeNonAscii(
        retry_call(
            repo.git.show,
            fargs=[f"{insert_commit}^:{insert_src_file}"],
            tries=5,
            delay=1,
        )
    )
    insert_src_file_path = os.path.join(top_dir, f"rq4/java/{i}_insert_src.java")

    insert_dst_file_content = removeNonAscii(
        retry_call(
            repo.git.show,
            fargs=[f"{insert_commit}:{insert_dst_file}"],
            tries=5,
            delay=1,
        )
    )
    insert_dst_file_path = os.path.join(top_dir, f"rq4/java/{i}_insert_dst.java")

    delete_src_file_content = removeNonAscii(
        retry_call(
            repo.git.show,
            fargs=[f"{delete_commit}^:{delete_src_file}"],
            tries=5,
            delay=1,
        )
    )
    delete_src_file_path = os.path.join(top_dir, f"rq4/java/{i}_delete_src.java")

    delete_dst_file_content = removeNonAscii(
        retry_call(
            repo.git.show,
            fargs=[f"{delete_commit}:{delete_dst_file}"],
            tries=5,
            delay=1,
        )
    )
    delete_dst_file_path = os.path.join(top_dir, f"rq4/java/{i}_delete_dst.java")

    with open(insert_src_file_path, "w") as f:
        f.write(insert_src_file_content)
    with open(insert_dst_file_path, "w") as f:
        f.write(insert_dst_file_content)
    with open(delete_src_file_path, "w") as f:
        f.write(delete_src_file_content)
    with open(delete_dst_file_path, "w") as f:
        f.write(delete_dst_file_content)

    process1 = subprocess.Popen(
        f"gumtree webdiff -g java-treesitter-ng --port 4567 {insert_src_file_path} {insert_dst_file_path}",
        shell=True,
        text=True,
    )
    process2 = subprocess.Popen(
        f"gumtree webdiff -g java-treesitter-ng --port 4568 {delete_src_file_path} {delete_dst_file_path}",
        shell=True,
        text=True,
    )

    print(
        URL_TEMPLATE.format(
            name_with_owner=name_with_owner.replace("_", "/", 1), commit=delete_commit
        )
    )

    process1.wait()
    process2.wait()

    break


  0%|          | 0/172 [00:00<?, ?it/s]

https://github.com/1and1/Troilus/commit/9c2a67349b9aba6da132587270f9603e2e5408b8


In [ ]:
import bisect


def calculate_line_offsets(source_code):
    """
    Calculates the start offsets for each line in the source code.

    Args:
        source_code (str): The full source code as a string.

    Returns:
        list[int]: A list of line start offsets.
    """
    offsets = [0]
    current_offset = 0

    for line in source_code.splitlines(keepends=True):
        current_offset += len(line)
        offsets.append(current_offset)

    return offsets


def position_for(offset, source_code):
    """
    Converts an offset into a (line, column) position based on the source code.

    Args:
        offset (int): The offset in the source code string.
        source_code (str): The full source code as a string.

    Returns:
        tuple: (line, column), both starting from 1.
    """
    lines = calculate_line_offsets(source_code)
    line = bisect.bisect_right(lines, offset) - 1
    column = offset - lines[line] + 1

    return line + 1, column


# サンプルのソースコード文字列
source_code = """public class Sample {
    public static void main(String[] args) {
        System.out.println("Hello, World!");
    }
}"""

# 使用例
offset = 22
print(calculate_line_offsets(source_code))
print(f"Offset: {offset}")
print("Position:", position_for(offset, source_code))

[0, 22, 67, 112, 118, 119]
Offset: 22
Position: (2, 1)


In [13]:
source_code[21]

'\n'

In [10]:
for i, line in enumerate(source_code.splitlines(keepends=True)):
    print(i, line)


0 public class Sample {

1     public static void main(String[] args) {

2         System.out.println("Hello, World!");

3     }

4 }


In [ ]:
/Users/kz/labo/study/fact_replacing_task_re/.venv/bin/python /Users/kz/labo/study/fact_replacing_task_re/rq4.py java -s 24